In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd

In [3]:
import pyspark.sql.functions as f
from pyspark.sql import SparkSession
from pyspark import SparkConf

In [4]:
parameters = {
    "spark.driver.maxResultSize": "3g",
    "spark.hadoop.fs.s3a.impl": "org.apache.hadoop.fs.s3a.S3AFileSystem",
    "spark.sql.execution.arrow.pyspark.enabled": True,

    # https://docs.kedro.org/en/stable/integrations/pyspark_integration.html#tips-for-maximising-concurrency-using-threadrunner
    "spark.scheduler.mode": "FAIR",
    "spark.driver.extraJavaOptions": "-Djava.security.manager=allow",
    "spark.executor.extraJavaOptions": "-Djava.security.manager=allow",

    "spark.sql.legacy.parquet.nanosAsLong": True,
    "spark.driver.memory": "40g",
}

spark_conf = SparkConf().setAll(parameters.items())
spark = SparkSession.builder.appName('New').enableHiveSupport().config(conf=spark_conf).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/11 11:44:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


25/04/11 11:44:26 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [6]:
int_transactions = spark.read.parquet("../data/02_intermediate/int_transactions")
prm_spine = spark.read.parquet("../data/03_primary/prm_spine")
prm_geolocation = spark.read.parquet("../data/04_feature/ftr_geolocation")

In [7]:
int_transactions.show()

prm_spine.show()

prm_geolocation.show()

25/04/11 11:47:12 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+--------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+
|      transaction_id|         customer_id| customer_vehicle_id|           branch_id|transaction_dt|      product_family|    product_category|product_sub_category|          product_id|is_fleet|is_pms|current_mileage|previous_mileage|new_customer|new_vehicle|warranty|quantity|net_cost|discount_amount|sales_amount_net|sales_amount|total_profit|sales_amount_net_perc|has_discount|delta_mileage|delta_mileage_perc|
+--------------------+--------------------+--------------------+--------------------+--------------+--------------------+--------------------+--------------------+-----------

In [8]:
transactions_data = int_transactions.withColumn(
    "_id",
    f.col("customer_id")
).withColumn(
    "_observ_end_dt",
    f.last_day(f.col("transaction_dt"))
)

In [9]:
base_sales = prm_spine.join(
    transactions_data,
    on=["_id", "_observ_end_dt"],
    how="left"
)

In [10]:
base_sales.show()

+--------------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------+--------------------+-------------------+--------------------+--------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+
|                 _id|_observ_end_dt|      transaction_id|         customer_id| customer_vehicle_id|           branch_id|transaction_dt|      product_family|   product_category|product_sub_category|          product_id|is_fleet|is_pms|current_mileage|previous_mileage|new_customer|new_vehicle|warranty|quantity|net_cost|discount_amount|sales_amount_net|sales_amount|total_profit|sales_amount_net_perc|has_discount|delta_mileage|delta_mileage_perc|
+--------------------+--------------+--------------------+--------------------+--------------------+----

In [11]:
sales_branch_view = base_sales.groupBy(
    ["_id", "_observ_end_dt", "branch_id"]
).agg(
    f.countDistinct("transaction_id").alias("trx_per_branch")
)

In [13]:
sales_branch_view.show(5, False)

+------------------------------------+--------------+------------------------------------+--------------+
|_id                                 |_observ_end_dt|branch_id                           |trx_per_branch|
+------------------------------------+--------------+------------------------------------+--------------+
|4CDAF1D5-512D-4EAE-931B-53F9DF678EBF|2024-01-31    |70919EBC-05BE-4CCF-8DF6-C6BCAC47BBA4|1             |
|7E9833DC-A403-4B2B-B267-0B1704B73A48|2024-06-30    |B1323649-5458-41A4-B03D-C584CE74AF3E|1             |
|076844A0-0120-4467-B805-272235724D56|2024-10-31    |FE6D471C-A328-49F1-A769-A2ADD53D24FC|1             |
|D3548A7A-8AE4-42A4-BBCF-1185F17B292A|2024-02-29    |1FCCE949-B97D-440C-BE1B-FFB03D0267CE|1             |
|CBA38EBA-F74C-45AE-B6A3-59D1ABA0966C|2024-05-31    |CEEFC64C-F2BA-48C9-AF65-1782F56B7D3E|1             |
+------------------------------------+--------------+------------------------------------+--------------+
only showing top 5 rows



In [18]:
sales_branch_geolocation = sales_branch_view.join(
    prm_geolocation.dropDuplicates(),
    on="branch_id",
    how="left"
)
sales_branch_geolocation.show(truncate=False)

+------------------------------------+------------------------------------+--------------+--------------+------------------+------------------+-----------+-----------+---------+--------+-------------------------------+-----------------------------------------------+-----------------------------------+--------------------------------+-------------------------------+----------------------------------+------------------------------+---------------------------------+------------------------------------+-------------------------+------------------------------+----------------------------------------------+----------------------------------+-------------------------------+------------------------------+---------------------------------+-----------------------------+--------------------------------+-----------------------------------+------------------------+------------------------------+----------------------------------------------+----------------------------------+-----------------------

In [19]:
from pyspark.sql import Window

In [22]:
w = Window.partitionBy('_id', '_observ_end_dt').orderBy(f.col('trx_per_branch').desc())

In [42]:
select_list = [
    f.col(col).alias(f"most_trx_branch_{col}")
    for col in geolocations_cols_to_keep
    if col not in ["_id", "_observ_end_dt",]
]

In [43]:
ftr_branch_most_trasactions_geolocation = sales_branch_geolocation.withColumn(
    'row_number',
    f.row_number().over(w)
).filter(
    f.col("trx_per_branch") > 0
).orderBy(
    "_id", "_observ_end_dt", "row_number"
).filter(
    f.col('row_number') == 1
).select("_id", "_observ_end_dt", *select_list)

ftr_branch_most_trasactions_geolocation.show(10)

+--------------------+--------------+-------------------------+-----------------------------------------------+---------------------------------------------------------------+---------------------------------------------------+------------------------------------------------+-----------------------------------------------+--------------------------------------------------+----------------------------------------------+-------------------------------------------------+----------------------------------------------------+-----------------------------------------+----------------------------------------------+--------------------------------------------------------------+--------------------------------------------------+-----------------------------------------------+----------------------------------------------+-------------------------------------------------+---------------------------------------------+------------------------------------------------+--------------------------------

In [32]:
geolocations_cols_to_drop = ["trx_per_branch", "longitude", "latitude","branch_code","branch_type","is_active","city"]
geolocations_cols_to_keep = [col for col in sales_branch_geolocation.columns if col not in geolocations_cols_to_drop]

In [35]:
agg_list = [f.avg(f.col(col)).alias(f"avg_{col}") for col in geolocations_cols_to_keep if col not in ["_id", "_observ_end_dt", "branch_id"]]

In [38]:

ftr_avg_branch_geolocation = sales_branch_geolocation.filter(
    f.col("trx_per_branch") > 0
).drop(
    *geolocations_cols_to_drop
).groupby(
    "_id", "_observ_end_dt"
).agg(*agg_list)

ftr_avg_branch_geolocation.show(truncate=False)

+------------------------------------+--------------+-----------------------------------+---------------------------------------------------+---------------------------------------+------------------------------------+-----------------------------------+--------------------------------------+----------------------------------+-------------------------------------+----------------------------------------+-----------------------------+----------------------------------+--------------------------------------------------+--------------------------------------+-----------------------------------+----------------------------------+-------------------------------------+---------------------------------+------------------------------------+---------------------------------------+----------------------------+----------------------------------+--------------------------------------------------+--------------------------------------+-----------------------------------+----------------------------

In [40]:
transactions_data.filter(
    f.col("_id") == "F23BA7E3-9FB6-4A93-8941-E92B8358F21B"
).filter(
    f.col("_observ_end_dt") == "2024-09-30"
).orderBy("_id", "_observ_end_dt").show(50, truncate=False)

+------------------------------------+------------------------------------+------------------------------------+------------------------------------+--------------+--------------------------------------+-----------------+--------------------+---------------------------------------------------------------------------------+--------+------+---------------+----------------+------------+-----------+--------+--------+--------+---------------+----------------+------------+------------+---------------------+------------+-------------+------------------+------------------------------------+--------------+
|transaction_id                      |customer_id                         |customer_vehicle_id                 |branch_id                           |transaction_dt|product_family                        |product_category |product_sub_category|product_id                                                                       |is_fleet|is_pms|current_mileage|previous_mileage|new_customer|new_vehic

In [41]:
prm_geolocation.filter(
    f.col("branch_id") == "EE5B9A50-31AF-4854-B259-6C8925347D21"
).show(50, truncate=False)

+------------------------------------+------------------+-----------------+-----------+-----------+---------+------+-------------------------------+-----------------------------------------------+-----------------------------------+--------------------------------+-------------------------------+----------------------------------+------------------------------+---------------------------------+------------------------------------+-------------------------+------------------------------+----------------------------------------------+----------------------------------+-------------------------------+------------------------------+---------------------------------+-----------------------------+--------------------------------+-----------------------------------+------------------------+------------------------------+----------------------------------------------+----------------------------------+-------------------------------+------------------------------+------------------------------

In [46]:
out = base_sales.select("_id", "_observ_end_dt").join(
    ftr_branch_most_trasactions_geolocation,
    on=["_id", "_observ_end_dt"],
    how="left"
).join(
    ftr_avg_branch_geolocation,
    on=["_id", "_observ_end_dt"],
    how="left"
).orderBy("_id", "_observ_end_dt")

In [47]:
out.show()

+--------------------+--------------+-------------------------+-----------------------------------------------+---------------------------------------------------------------+---------------------------------------------------+------------------------------------------------+-----------------------------------------------+--------------------------------------------------+----------------------------------------------+-------------------------------------------------+----------------------------------------------------+-----------------------------------------+----------------------------------------------+--------------------------------------------------------------+--------------------------------------------------+-----------------------------------------------+----------------------------------------------+-------------------------------------------------+---------------------------------------------+------------------------------------------------+--------------------------------